# Notebook 04: Solar polygon footprint

In [1]:
SECTOR = "Solar"
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

# I use the verified Notebook 03 exports without repeating the county spatial joins.
ROOT = Path(r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint")
INPUT = ROOT / "05_Optimized_Data" / "GeoParquet"
OUTPUT = ROOT / "06_Energy_Footprint_Analysis" / SECTOR
OUTPUT.mkdir(parents=True, exist_ok=True)
ACRE_M2 = 4046.8564224
FT_PER_MILE = 5280.0
SQFT_PER_ACRE = 43560.0
SCENARIOS = ("low", "central", "high")

def read_layer(name):
    path = INPUT / f"{name}.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 03 export: {path}")
    data = gpd.read_parquet(path)
    if data.crs is None:
        raise ValueError(f"{name}: missing CRS")
    if data.geometry.isna().any() or data.geometry.is_empty.any():
        raise ValueError(f"{name}: missing or empty geometries")
    return data

def county_key(df):
    if "county_geoid" not in df:
        raise KeyError("Notebook 03 county_geoid is required")
    s = df["county_geoid"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    return s.where(s.str.fullmatch(r"48\d{3}"), pd.NA)

def save_results(detail, county, state, assumptions):
    # I save both geographic levels and verify that each scenario reconciles.
    for scenario in SCENARIOS:
        col = f"{scenario}_acres"
        if col not in detail or col not in county or col not in state:
            raise KeyError(f"Missing scenario output: {col}")
        a = float(detail[col].sum()); b = float(county[col].sum()); c = float(state[col].sum())
        if not np.isclose(a, c, rtol=1e-9, atol=0.01):
            raise AssertionError(f"Statewide {scenario} does not match facility detail")
        # I report unassigned county acreage separately rather than force a county allocation.
        unassigned = float(detail.loc[detail['county_geoid'].isna(), col].sum())
        if not np.isclose(b + unassigned, c, rtol=1e-9, atol=0.01):
            raise AssertionError(f"County plus unassigned acreage does not reconcile: {scenario}")
    detail.drop(columns=['geometry'], errors='ignore').to_csv(OUTPUT / 'facility_scenarios.csv', index=False)
    county.to_csv(OUTPUT / 'county_scenarios.csv', index=False)
    state.to_csv(OUTPUT / 'statewide_scenarios.csv', index=False)
    assumptions.to_csv(OUTPUT / 'scenario_assumptions.csv', index=False)
    print('Saved:', OUTPUT)
    display(state.round(2))
    display(county.head(10).round(2))


In [2]:
parts = read_layer("Texas_Solar_County_Project_Polygons")
unique = read_layer("Texas_Solar_County_Unique_Footprint")
assert len(parts)==197 and parts["solar_record_id"].nunique()==187
parts["county_geoid"] = county_key(parts)
unique["county_geoid"] = county_key(unique)
assert parts["county_geoid"].notna().all() and unique["county_geoid"].notna().all()
assert unique["county_geoid"].is_unique
assert parts.crs.is_projected and unique.crs.is_projected
parts["project_acres_recalculated"] = parts.geometry.area / ACRE_M2
unique["unique_acres_recalculated"] = unique.geometry.area / ACRE_M2
assert np.isclose(parts["project_acres_recalculated"].sum(), 143025.17, atol=1.0)
print("Summed project polygon acres:", parts["project_acres_recalculated"].sum())
print("Unique polygon acres:", unique["unique_acres_recalculated"].sum())

Summed project polygon acres: 143025.17307269835
Unique polygon acres: 143025.17307269835


## Assumptions
The **central** scenario is 100% of mapped unique polygon acreage, which is a mapped-site-area measure rather than verified permanent disturbance. The **low** and **high** sensitivity scenarios use 90% and 110% of mapped unique acreage. The high case is an extrapolated site-area sensitivity, not additional mapped polygon area. These percentages are illustrative and require independent justification before publication.

In [3]:
assumptions = pd.DataFrame({"scenario":list(SCENARIOS),"mapped_area_multiplier":[0.9,1.0,1.1],"basis":["Exploratory 90% site-area sensitivity","Measured union of mapped polygons","Exploratory 110% site-area sensitivity"]})
solar = unique.copy()
for s, multiplier in zip(SCENARIOS, assumptions["mapped_area_multiplier"]):
    solar[f"{s}_acres"] = solar["unique_acres_recalculated"] * multiplier
county = solar[["county_geoid","unique_acres_recalculated"]+[f"{s}_acres" for s in SCENARIOS]].copy()
project_by_county = parts.groupby("county_geoid",as_index=False).agg(project_polygon_acres=("project_acres_recalculated","sum"),project_parts=("solar_record_id","size"),distinct_projects=("solar_record_id","nunique"))
county = county.merge(project_by_county,on="county_geoid",how="left",validate="one_to_one")
assert (county["unique_acres_recalculated"] <= county["project_polygon_acres"]+0.1).all()
state = pd.DataFrame([{"sector":"Solar","original_projects":187,"county_parts":len(parts),"mapped_project_polygon_acres":parts["project_acres_recalculated"].sum(),"mapped_unique_acres":solar["unique_acres_recalculated"].sum(),**{f"{s}_acres":solar[f"{s}_acres"].sum() for s in SCENARIOS}}])
save_results(solar,county,state,assumptions)

Saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Solar


,sector,original_projects,county_parts,mapped_project_polygon_acres,mapped_unique_acres,low_acres,central_acres,high_acres
0,Solar,187,197,143025.17,143025.17,128722.66,143025.17,157327.69


,county_geoid,unique_acres_recalculated,low_acres,central_acres,high_acres,project_polygon_acres,project_parts,distinct_projects
0,48481,5747.79,5173.01,5747.79,6322.57,5747.79,5,5
1,48303,27.32,24.59,27.32,30.05,27.32,1,1
2,48157,6776.13,6098.52,6776.13,7453.75,6776.13,5,5
3,48015,59.20,53.28,59.20,65.12,59.20,2,2
4,48201,4.21,3.78,4.21,4.63,4.21,1,1
5,48039,5077.39,4569.65,5077.39,5585.13,5077.39,6,6
6,48291,681.55,613.39,681.55,749.70,681.55,1,1
7,48099,105.61,95.05,105.61,116.18,105.61,1,1
8,48027,5949.86,5354.88,5949.86,6544.85,5949.86,3,3
9,48227,50.71,45.64,50.71,55.78,50.71,1,1


In [4]:
# I calculate the mapped solar footprint and three sensitivity scenarios
# for all 254 Texas counties.

from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT = Path(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)

# I retain the measured mapped area as the central scenario.
scenario_rates = {
    "low": 0.90,
    "central": 1.00,
    "high": 1.10
}

assert set(SCENARIOS) == set(scenario_rates)

assumptions = pd.DataFrame([
    {
        "scenario": scenario,
        "mapped_area_multiplier": multiplier,
        "basis": (
            "Measured union of mapped polygons"
            if scenario == "central"
            else "Exploratory mapped-area sensitivity"
        )
    }
    for scenario, multiplier in scenario_rates.items()
])

# I preserve the original unique solar footprint data.
solar = unique.copy()
project_parts = parts.copy()

# I verify the required columns before calculating acreage.
assert {
    "county_geoid",
    "unique_acres_recalculated"
}.issubset(solar.columns)

assert {
    "county_geoid",
    "solar_record_id",
    "project_acres_recalculated"
}.issubset(project_parts.columns)

# I standardize county GEOIDs without assigning missing locations.
for df in [solar, project_parts]:
    df["county_geoid"] = (
        df["county_geoid"]
        .astype("string")
        .str.strip()
        .str.zfill(5)
    )

    df.loc[
        df["county_geoid"].isin(["<NA>", "nan", "None", ""]),
        "county_geoid"
    ] = pd.NA

# I verify that measured acreage is valid.
assert solar["unique_acres_recalculated"].notna().all()
assert (solar["unique_acres_recalculated"] >= 0).all()

assert project_parts["project_acres_recalculated"].notna().all()
assert (project_parts["project_acres_recalculated"] >= 0).all()

# I calculate acreage under the three scenarios.
for scenario, multiplier in scenario_rates.items():
    solar[f"{scenario}_acres"] = (
        solar["unique_acres_recalculated"] * multiplier
    )

# I aggregate unique mapped solar acreage by county.
county = (
    solar.loc[solar["county_geoid"].notna()]
    .groupby("county_geoid", as_index=False)
    .agg(
        unique_acres_recalculated=(
            "unique_acres_recalculated", "sum"
        ),
        **{
            f"{scenario}_acres": (
                f"{scenario}_acres", "sum"
            )
            for scenario in SCENARIOS
        }
    )
)

# I calculate project polygon acreage and project counts separately.
project_by_county = (
    project_parts.loc[
        project_parts["county_geoid"].notna()
    ]
    .groupby("county_geoid", as_index=False)
    .agg(
        project_polygon_acres=(
            "project_acres_recalculated", "sum"
        ),
        project_parts=("solar_record_id", "size"),
        distinct_projects=("solar_record_id", "nunique")
    )
)

county = county.merge(
    project_by_county,
    on="county_geoid",
    how="outer",
    validate="one_to_one"
)

# I verify that both solar datasets cover the same counties.
assert county["unique_acres_recalculated"].notna().all()
assert county["project_polygon_acres"].notna().all()

# I verify that unique footprint does not materially exceed
# the sum of project polygon areas.
assert (
    county["unique_acres_recalculated"]
    <= county["project_polygon_acres"] + 0.1
).all()

# I calculate statewide totals using the complete mapped datasets.
state = pd.DataFrame([{
    "sector": "Solar",
    "original_projects": project_parts["solar_record_id"].nunique(),
    "county_parts": len(project_parts),
    "mapped_project_polygon_acres": (
        project_parts["project_acres_recalculated"].sum()
    ),
    "mapped_unique_acres": (
        solar["unique_acres_recalculated"].sum()
    ),
    **{
        f"{scenario}_acres": solar[f"{scenario}_acres"].sum()
        for scenario in SCENARIOS
    }
}])

# I retrieve the official Texas county names and GEOIDs.
census_url = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)

county_reference = pd.read_csv(
    census_url,
    sep="|",
    dtype=str,
    encoding="utf-8-sig"
)

county_reference = county_reference.loc[
    county_reference["STATEFP"] == "48",
    ["STATEFP", "COUNTYFP", "COUNTYNAME"]
].copy()

county_reference["county_geoid"] = (
    county_reference["STATEFP"]
    + county_reference["COUNTYFP"]
)

county_reference = county_reference.rename(
    columns={"COUNTYNAME": "county_name"}
)[["county_geoid", "county_name"]]

# I verify the county reference before merging.
assert len(county_reference) == 254
assert county_reference["county_geoid"].is_unique

# I check that all mapped solar counties have valid Texas GEOIDs.
unrecognized = (
    set(county["county_geoid"])
    - set(county_reference["county_geoid"])
)

assert not unrecognized, (
    f"Unrecognized county GEOIDs: {sorted(unrecognized)}"
)

# I identify counties without mapped solar facilities.
counties_without_solar = county_reference.loc[
    ~county_reference["county_geoid"].isin(county["county_geoid"])
].copy()

# I retain all 254 Texas counties in the final results.
county_complete = county_reference.merge(
    county,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

numeric_columns = [
    "unique_acres_recalculated",
    "low_acres",
    "central_acres",
    "high_acres",
    "project_polygon_acres",
    "project_parts",
    "distinct_projects"
]

county_complete[numeric_columns] = (
    county_complete[numeric_columns].fillna(0)
)

# I preserve integer counts for projects and county pieces.
county_complete["project_parts"] = (
    county_complete["project_parts"].astype(int)
)

county_complete["distinct_projects"] = (
    county_complete["distinct_projects"].astype(int)
)

# I sort the complete county table alphabetically.
county_complete = (
    county_complete
    .sort_values("county_name")
    .reset_index(drop=True)
)

# I verify that all Texas counties are represented.
assert len(county_complete) == 254
assert county_complete["county_geoid"].is_unique

# I keep unassigned geographic records separate rather than
# allocating their acreage to an arbitrary county.
unassigned_unique = solar.loc[
    solar["county_geoid"].isna()
].copy()

unassigned_parts = project_parts.loc[
    project_parts["county_geoid"].isna()
].copy()

# I reconcile county acreage and unassigned acreage with statewide totals.
assert np.isclose(
    county_complete["unique_acres_recalculated"].sum()
    + unassigned_unique["unique_acres_recalculated"].sum(),
    state["mapped_unique_acres"].iloc[0],
    rtol=0,
    atol=0.01
)

assert np.isclose(
    county_complete["project_polygon_acres"].sum()
    + unassigned_parts["project_acres_recalculated"].sum(),
    state["mapped_project_polygon_acres"].iloc[0],
    rtol=0,
    atol=0.01
)

for scenario in SCENARIOS:
    assert np.isclose(
        county_complete[f"{scenario}_acres"].sum()
        + unassigned_unique[f"{scenario}_acres"].sum(),
        state[f"{scenario}_acres"].iloc[0],
        rtol=0,
        atol=0.01
    ), f"Reconciliation failed for {scenario}"

# I calculate the overlap diagnostic without rounding the source values.
overlap_difference = (
    state["mapped_project_polygon_acres"].iloc[0]
    - state["mapped_unique_acres"].iloc[0]
)

# I export the detailed, statewide, county, and assumptions results.
save_results(
    solar,
    county_complete,
    state,
    assumptions
)

# I save the complete county table with names.
county_complete.to_csv(
    OUTPUT / "solar_county_254_with_names.csv",
    index=False
)

# I preserve the project polygon counts and areas for data checking.
project_by_county.to_csv(
    OUTPUT / "solar_project_county_audit.csv",
    index=False
)

# I report the final validation results.
print("\nFINAL SOLAR VALIDATION")
print("Total Texas counties:", len(county_complete))
print(
    "Counties with mapped solar:",
    (county_complete["unique_acres_recalculated"] > 0).sum()
)
print(
    "Counties without mapped solar:",
    (county_complete["unique_acres_recalculated"] == 0).sum()
)
print("Original solar projects:", state["original_projects"].iloc[0])
print("County-clipped project pieces:", len(project_parts))
print(
    "Mapped unique acreage:",
    f"{state['mapped_unique_acres'].iloc[0]:,.2f}"
)
print(
    "Project polygon acreage:",
    f"{state['mapped_project_polygon_acres'].iloc[0]:,.2f}"
)
print(
    "Project area minus unique area:",
    f"{overlap_difference:,.6f} acres"
)

for scenario in SCENARIOS:
    print(
        f"{scenario.capitalize()} acreage:",
        f"{state[f'{scenario}_acres'].iloc[0]:,.2f}"
    )

print("\nPASS: Solar county and statewide acreage reconcile.")

display(state)
display(county_complete)

Saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Solar


,sector,original_projects,county_parts,mapped_project_polygon_acres,mapped_unique_acres,low_acres,central_acres,high_acres
0,Solar,187,197,143025.17,143025.17,128722.66,143025.17,157327.69


,county_geoid,county_name,unique_acres_recalculated,low_acres,central_acres,high_acres,project_polygon_acres,project_parts,distinct_projects
0,48001,Anderson County,0.0,0.00,0.0,0.00,0.0,0,0
1,48003,Andrews County,7202.8,6482.52,7202.8,7923.08,7202.8,4,4
2,48005,Angelina County,0.0,0.00,0.0,0.00,0.0,0,0
3,48007,Aransas County,0.0,0.00,0.0,0.00,0.0,0,0
4,48009,Archer County,0.0,0.00,0.0,0.00,0.0,0,0
5,48011,Armstrong County,0.0,0.00,0.0,0.00,0.0,0,0
6,48013,Atascosa County,0.0,0.00,0.0,0.00,0.0,0,0
7,48015,Austin County,59.2,53.28,59.2,65.12,59.2,2,2
8,48017,Bailey County,0.0,0.00,0.0,0.00,0.0,0,0
9,48019,Bandera County,0.0,0.00,0.0,0.00,0.0,0,0



FINAL SOLAR VALIDATION
Total Texas counties: 254
Counties with mapped solar: 86
Counties without mapped solar: 168
Original solar projects: 187
County-clipped project pieces: 197
Mapped unique acreage: 143,025.17
Project polygon acreage: 143,025.17
Project area minus unique area: 0.000000 acres
Low acreage: 128,722.66
Central acreage: 143,025.17
High acreage: 157,327.69

PASS: Solar county and statewide acreage reconcile.


,sector,original_projects,county_parts,mapped_project_polygon_acres,mapped_unique_acres,low_acres,central_acres,high_acres
0,Solar,187,197,143025.173073,143025.173073,128722.655765,143025.173073,157327.69038


,county_geoid,county_name,unique_acres_recalculated,low_acres,central_acres,high_acres,project_polygon_acres,project_parts,distinct_projects
0,48001,Anderson County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
1,48003,Andrews County,7202.802984,6482.522685,7202.802984,7923.083282,7202.802984,4,4
2,48005,Angelina County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
3,48007,Aransas County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
4,48009,Archer County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
...,...,...,...,...,...,...,...,...,...
249,48499,Wood County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
250,48501,Yoakum County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
251,48503,Young County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
252,48505,Zapata County,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
